# 1. Data Load

In [769]:
import os
from pathlib import Path
import numpy as np

In [770]:
data_path = Path("../data")

movie_path = f"{data_path}/movies.csv"
rating_path = f"{data_path}/ratings.csv"
tag_path = f"{data_path}/tags.csv"
link_path = f"{data_path}/links.csv"

In [771]:
import pandas as pd

movies_df = pd.read_csv(movie_path)
ratings_df = pd.read_csv(rating_path)
tags_df = pd.read_csv(tag_path)
links_df = pd.read_csv(link_path)


## a) Null check

In [772]:
df_list = [movies_df, ratings_df, tags_df]

for df in df_list:
    print(df.isnull().sum(), '\n')

movieId    0
title      0
genres     0
dtype: int64 

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64 

userId       0
movieId      0
tag          0
timestamp    0
dtype: int64 



# b) index check

In [773]:
movies_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  9742 non-null   int64 
 1   title    9742 non-null   object
 2   genres   9742 non-null   object
dtypes: int64(1), object(2)
memory usage: 228.5+ KB


In [774]:
ratings_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


In [775]:
tags_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3683 entries, 0 to 3682
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   userId     3683 non-null   int64 
 1   movieId    3683 non-null   int64 
 2   tag        3683 non-null   object
 3   timestamp  3683 non-null   int64 
dtypes: int64(3), object(1)
memory usage: 115.2+ KB


In [776]:
links_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   movieId  9742 non-null   int64  
 1   imdbId   9742 non-null   int64  
 2   tmdbId   9734 non-null   float64
dtypes: float64(1), int64(2)
memory usage: 228.5 KB


movie id, user id 는 속도를 위해 int 형으로 그대로 사용했습니다.

# 장르 결측치 처리

https://www.themoviedb.org/

# a) Check null genre

In [777]:
print(movies_df['genres'].str.contains('no genres listed').sum(),"개의 장르 결측치 존재")

34 개의 장르 결측치 존재


In [778]:
movies_df['genres']

0       Adventure|Animation|Children|Comedy|Fantasy
1                        Adventure|Children|Fantasy
2                                    Comedy|Romance
3                              Comedy|Drama|Romance
4                                            Comedy
                           ...                     
9737                Action|Animation|Comedy|Fantasy
9738                       Animation|Comedy|Fantasy
9739                                          Drama
9740                               Action|Animation
9741                                         Comedy
Name: genres, Length: 9742, dtype: object

In [779]:
genres_null = movies_df[movies_df['genres'].str.contains('no genres listed')]
genres_null

,movieId,title,genres
8517,114335,La cravate (1957),(no genres listed)
8684,122888,Ben-hur (2016),(no genres listed)
8687,122896,Pirates of the Caribbean: Dead Men Tell No Tales (2017),(no genres listed)
8782,129250,Superfast! (2015),(no genres listed)
8836,132084,Let It Be Me (1995),(no genres listed)
8902,134861,Trevor Noah: African American (2013),(no genres listed)
9033,141131,Guardians (2016),(no genres listed)
9053,141866,Green Room (2015),(no genres listed)
9070,142456,The Brand New Testament (2015),(no genres listed)
9091,143410,Hyena Road,(no genres listed)


In [780]:
pd.set_option('display.max_colwidth', None)
genres_null

,movieId,title,genres
8517,114335,La cravate (1957),(no genres listed)
8684,122888,Ben-hur (2016),(no genres listed)
8687,122896,Pirates of the Caribbean: Dead Men Tell No Tales (2017),(no genres listed)
8782,129250,Superfast! (2015),(no genres listed)
8836,132084,Let It Be Me (1995),(no genres listed)
8902,134861,Trevor Noah: African American (2013),(no genres listed)
9033,141131,Guardians (2016),(no genres listed)
9053,141866,Green Room (2015),(no genres listed)
9070,142456,The Brand New Testament (2015),(no genres listed)
9091,143410,Hyena Road,(no genres listed)


# b) fill genre manually
- fine all genre
- fine movie's genre from internet

In [781]:
exist_genre_index = movies_df.index.difference(genres_null.index)
all_genres = movies_df.loc[exist_genre_index]['genres'].replace(r'\|', ' ', regex=True).str.split(' ').explode().unique()

In [782]:
all_genres

array(['Adventure', 'Animation', 'Children', 'Comedy', 'Fantasy',
       'Romance', 'Drama', 'Action', 'Crime', 'Thriller', 'Horror',
       'Mystery', 'Sci-Fi', 'War', 'Musical', 'Documentary', 'IMAX',
       'Western', 'Film-Noir'], dtype=object)

In [783]:
movies_df.loc[movies_df['title']=='La cravate (1957)', 'genres']='Fantasy|Comedy'

In [784]:
movies_df[movies_df['title'] == 'La cravate (1957)']

,movieId,title,genres
8517,114335,La cravate (1957),Fantasy|Comedy


In [785]:
movies_df.loc[movies_df['title']=='Ben-hur (2016)', 'genres'] = 'Action|Adventure|Drama'
movies_df.loc[movies_df['title']=='Pirates of the Caribbean: Dead Men Tell No Tales (2017)', 'genres'] = 'Action|Adventure|Fantasy'
movies_df.loc[movies_df['title']=='Superfast! (2015)', 'genres'] = 'Comedy|Action'
movies_df.loc[movies_df['title']=='Let It Be Me (1995)', 'genres'] = 'Romance'
movies_df.loc[movies_df['title']=='Trevor Noah: African American (2013)', 'genres'] = 'Comedy'
movies_df.loc[movies_df['title']=='Guardians (2016)', 'genres'] = 'Action Adventure Drama' # 확인 필요

# 김태연님
movies_df.loc[movies_df['title']=='The Adventures of Sherlock Holmes and Doctor Watson: The Hunt for the Tiger (1980)', 'genres'] = 'Mystery|Crime|Adventure'
movies_df.loc[movies_df['title']=='The Putin Interviews (2017)', 'genres'] = 'Documentary|War'
movies_df.loc[movies_df['title']=='Black Mirror', 'genres'] = 'Sci-Fi|Fantasy|Drama|Mystery'
movies_df.loc[movies_df['title']=='Too Funny to Fail: The Life and Death of The Dana Carvey Show (2017)', 'genres'] = 'Documentary'
movies_df.loc[movies_df['title']=='Serving in Silence: The Margarethe Cammermeyer Story (1995)', 'genres'] = 'Drama'
movies_df.loc[movies_df['title']=='A Christmas Story Live! (2017)', 'genres'] = 'Musical'
# 홍승혜님
movies_df.loc[movies_df['title']=='Cosmos', 'genres'] = 'Documentary'
movies_df.loc[movies_df['title']=='Maria Bamford: Old Baby', 'genres'] = 'Comedy|Documentary'
movies_df.loc[movies_df['title']=='Death Note: Desu nôto (2006-2007)', 'genres'] = 'Animation|Mystery|Sci-Fi|Fantasy'
movies_df.loc[movies_df['title']=='Generation Iron 2', 'genres'] = 'Documentary'
movies_df.loc[movies_df['title']=='T2 3-D: Battle Across Time (1996)', 'genres'] = 'Action|Sci-Fi'
movies_df.loc[movies_df['title']=='The Godfather Trilogy: 1972-1990 (1992)	', 'genres'] = 'Crime|Drama'

movies_df.loc[movies_df['title']=='A Cosmic Christmas (1977)', 'genres'] = 'Animation|Sci-Fi'
movies_df.loc[movies_df['title']=='Green Room (2015)', 'genres'] = 'Action|Horror'
movies_df.loc[movies_df['title']=='The Brand New Testament (2015)', 'genres'] = 'Comedy|Fantasy'
movies_df.loc[movies_df['title']=='Hyena Road', 'genres'] = 'War|Drama|Thriller|Action'
movies_df.loc[movies_df['title']=='The Adventures of Sherlock Holmes and Doctor Watson', 'genres'] = 'Mystery|Crime|Adventure'

movies_df.loc[movies_df['title']=='Grease Live (2016)', 'genres'] = 'Musical'
movies_df.loc[movies_df['title']=='Noin 7 veljestä (1968)', 'genres'] = 'Comedy'
movies_df.loc[movies_df['title']=='Paterson', 'genres'] = 'Drama|Comedy|Romance'
movies_df.loc[movies_df['title']=='Ali Wong: Baby Cobra (2016)', 'genres'] = 'Comedy'
movies_df.loc[movies_df['title']=="A Midsummer Night's Dream (2016)", 'genres'] = 'Comedy|Romance|Fantasy'
movies_df.loc[movies_df['title']=='The Forbidden Dance (1990)', 'genres'] = 'Romance|Drama'
movies_df.loc[movies_df['title']=='Ethel & Ernest (2016)', 'genres'] = 'Animation|Drama|War'
movies_df.loc[movies_df['title']=='Whiplash (2013)', 'genres'] = 'Drama|Musical'
movies_df.loc[movies_df['title']=='The OA', 'genres'] = 'Mystery|Drama'
movies_df.loc[movies_df['title']=='Lemonade (2016)', 'genres'] = 'Musical'
movies_df.loc[movies_df['title']=='Death Note: Desu nôto (2006–2007)', 'genres'] = 'Animation|Mystery|Sci-Fi|Fantasy'
movies_df.loc[movies_df['title']=='The Godfather Trilogy: 1972-1990 (1992)', 'genres'] = 'Drama|Crime'


In [786]:
temp_df = movies_df.copy() # 코드 재실행시를 위한 temp df 생성

In [787]:
# check successful filling missing value in genres
movies_df.loc[genres_null.index]

,movieId,title,genres
8517,114335,La cravate (1957),Fantasy|Comedy
8684,122888,Ben-hur (2016),Action|Adventure|Drama
8687,122896,Pirates of the Caribbean: Dead Men Tell No Tales (2017),Action|Adventure|Fantasy
8782,129250,Superfast! (2015),Comedy|Action
8836,132084,Let It Be Me (1995),Romance
8902,134861,Trevor Noah: African American (2013),Comedy
9033,141131,Guardians (2016),Action Adventure Drama
9053,141866,Green Room (2015),Action|Horror
9070,142456,The Brand New Testament (2015),Comedy|Fantasy
9091,143410,Hyena Road,War|Drama|Thriller|Action


# title duplicated

# a) split genres for combine

In [788]:
pd.set_option('display.max_colwidth',None)
movies_df['genres'] = temp_df['genres'].replace(r'\|', ' ', regex=True).str.split()
movies_df

,movieId,title,genres
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]"
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]"
2,3,Grumpier Old Men (1995),"[Comedy, Romance]"
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]"
4,5,Father of the Bride Part II (1995),[Comedy]
...,...,...,...
9737,193581,Black Butler: Book of the Atlantic (2017),"[Action, Animation, Comedy, Fantasy]"
9738,193583,No Game No Life: Zero (2017),"[Animation, Comedy, Fantasy]"
9739,193585,Flint (2017),[Drama]
9740,193587,Bungo Stray Dogs: Dead Apple (2018),"[Action, Animation]"


# b) title duplicated check
- 연도와 영화 이름까지 같은 경우

In [789]:
duplicated_title = movies_df[movies_df['title'].duplicated()]['title']
movies_df[movies_df['title'].duplicated()]

,movieId,title,genres
5601,26958,Emma (1996),[Romance]
6932,64997,War of the Worlds (2005),"[Action, Sci-Fi]"
9106,144606,Confessions of a Dangerous Mind (2002),"[Comedy, Crime, Drama, Romance, Thriller]"
9135,147002,Eros (2004),"[Drama, Romance]"
9468,168358,Saturn 3 (1980),"[Sci-Fi, Thriller]"


In [790]:
dup_temp = movies_df[movies_df['title'].isin(duplicated_title)].sort_values(by='title') # 가시성을 위함
dup_temp

,movieId,title,genres
4169,6003,Confessions of a Dangerous Mind (2002),"[Comedy, Crime, Drama, Thriller]"
9106,144606,Confessions of a Dangerous Mind (2002),"[Comedy, Crime, Drama, Romance, Thriller]"
650,838,Emma (1996),"[Comedy, Drama, Romance]"
5601,26958,Emma (1996),[Romance]
5854,32600,Eros (2004),[Drama]
9135,147002,Eros (2004),"[Drama, Romance]"
2141,2851,Saturn 3 (1980),"[Adventure, Sci-Fi, Thriller]"
9468,168358,Saturn 3 (1980),"[Sci-Fi, Thriller]"
5931,34048,War of the Worlds (2005),"[Action, Adventure, Sci-Fi, Thriller]"
6932,64997,War of the Worlds (2005),"[Action, Sci-Fi]"


In [791]:
links_df[links_df['movieId'].isin(dup_temp['movieId'].values)]

,movieId,imdbId,tmdbId
650,838,116191,3573.0
2141,2851,81454,NaN
4169,6003,290538,4912.0
5601,26958,118308,12254.0
5854,32600,377059,NaN
5931,34048,407304,74.0
6932,64997,449040,34812.0
9106,144606,270288,4912.0
9135,147002,343663,39850.0
9468,168358,79285,19761.0


### **saturn3 / eros / confessions of a dangerous mind 는 동일 영화** , IMDb 에서 확인

In [792]:
movies_df.loc[movies_df['movieId']==64997,'title'] = 'War of the Worlds [2] (2005)'
movies_df.loc[movies_df['movieId']==26958,'title'] = 'Emma [2] (1996)'

In [793]:
movies_df[movies_df['movieId'].isin(dup_temp['movieId'])] # Emma / War of the Worlds check

,movieId,title,genres
650,838,Emma (1996),"[Comedy, Drama, Romance]"
2141,2851,Saturn 3 (1980),"[Adventure, Sci-Fi, Thriller]"
4169,6003,Confessions of a Dangerous Mind (2002),"[Comedy, Crime, Drama, Thriller]"
5601,26958,Emma [2] (1996),[Romance]
5854,32600,Eros (2004),[Drama]
5931,34048,War of the Worlds (2005),"[Action, Adventure, Sci-Fi, Thriller]"
6932,64997,War of the Worlds [2] (2005),"[Action, Sci-Fi]"
9106,144606,Confessions of a Dangerous Mind (2002),"[Comedy, Crime, Drama, Romance, Thriller]"
9135,147002,Eros (2004),"[Drama, Romance]"
9468,168358,Saturn 3 (1980),"[Sci-Fi, Thriller]"


In [794]:
duplicated_title = movies_df[movies_df['title'].duplicated()]['title']
dup_temp = movies_df[movies_df['title'].isin(duplicated_title)].sort_values(by='title') # 가시성을 위함
dup_temp

,movieId,title,genres
4169,6003,Confessions of a Dangerous Mind (2002),"[Comedy, Crime, Drama, Thriller]"
9106,144606,Confessions of a Dangerous Mind (2002),"[Comedy, Crime, Drama, Romance, Thriller]"
5854,32600,Eros (2004),[Drama]
9135,147002,Eros (2004),"[Drama, Romance]"
2141,2851,Saturn 3 (1980),"[Adventure, Sci-Fi, Thriller]"
9468,168358,Saturn 3 (1980),"[Sci-Fi, Thriller]"


In [795]:
# 장르를 합치기 위한 union
group_temp = dup_temp.groupby(by='title')
union_genre = group_temp['genres'].apply(
    lambda x: list(set.union(*map(set, x)))
)

union_genre

title
Confessions of a Dangerous Mind (2002)    [Crime, Drama, Romance, Comedy, Thriller]
Eros (2004)                                                        [Drama, Romance]
Saturn 3 (1980)                                       [Thriller, Adventure, Sci-Fi]
Name: genres, dtype: object

In [796]:
use_idx = group_temp['genres'].idxmax()
use_idx

title
Confessions of a Dangerous Mind (2002)    4169
Eros (2004)                               9135
Saturn 3 (1980)                           9468
Name: genres, dtype: int64

In [797]:
drop_idx = dup_temp.index.difference(use_idx)
dup_temp.loc[drop_idx]

,movieId,title,genres
2141,2851,Saturn 3 (1980),"[Adventure, Sci-Fi, Thriller]"
5854,32600,Eros (2004),[Drama]
9106,144606,Confessions of a Dangerous Mind (2002),"[Comedy, Crime, Drama, Romance, Thriller]"


# c) replace genre

In [798]:
dup_temp.loc[use_idx]['genres'] =  dup_temp['title'].map(union_genre) # use_idx 에는 합집합 연산을 한 장르를 넣어줌
dup_temp

,movieId,title,genres
4169,6003,Confessions of a Dangerous Mind (2002),"[Comedy, Crime, Drama, Thriller]"
9106,144606,Confessions of a Dangerous Mind (2002),"[Comedy, Crime, Drama, Romance, Thriller]"
5854,32600,Eros (2004),[Drama]
9135,147002,Eros (2004),"[Drama, Romance]"
2141,2851,Saturn 3 (1980),"[Adventure, Sci-Fi, Thriller]"
9468,168358,Saturn 3 (1980),"[Sci-Fi, Thriller]"


In [799]:
drop_movieId = dup_temp.loc[drop_idx]['movieId'] # 다른 DB 에서 movieId 를 최신화 하기 위해서 movieId 추출
max_len_movieId = dup_temp.loc[use_idx]['movieId']

In [800]:
movies_refine = movies_df.drop(drop_idx).copy()

In [801]:
print("삭제된 ROW :\t", abs(len(movies_df) - len(movies_refine)))
print("dup index 갯수 :\t", len(drop_idx))

삭제된 ROW :	 3
dup index 갯수 :	 3


In [802]:
movies_refine.loc[use_idx]

,movieId,title,genres
4169,6003,Confessions of a Dangerous Mind (2002),"[Comedy, Crime, Drama, Thriller]"
9135,147002,Eros (2004),"[Drama, Romance]"
9468,168358,Saturn 3 (1980),"[Sci-Fi, Thriller]"


In [803]:
max_len_movieId

4169      6003
9135    147002
9468    168358
Name: movieId, dtype: int64

In [804]:
# 중복 제거 확인용 (1) movieId 값을 이용해서 확인
movies_refine[movies_refine['movieId'].isin(max_len_movieId.values)]

,movieId,title,genres
4169,6003,Confessions of a Dangerous Mind (2002),"[Comedy, Crime, Drama, Thriller]"
9135,147002,Eros (2004),"[Drama, Romance]"
9468,168358,Saturn 3 (1980),"[Sci-Fi, Thriller]"


In [805]:
# 중복 제거 확인용 (2)
movies_refine[movies_refine['movieId'].isin(drop_movieId.values)]

,movieId,title,genres


In [806]:
movies_df = movies_refine

In [807]:
max_len_movieId.values

array([  6003, 147002, 168358])

In [808]:
idx_list = group_temp['movieId'].apply(list)
idx_list

title
Confessions of a Dangerous Mind (2002)     [6003, 144606]
Eros (2004)                               [32600, 147002]
Saturn 3 (1980)                            [2851, 168358]
Name: movieId, dtype: object

In [809]:
# mapping 을 위한 series 생성
map_series = dup_temp.loc[use_idx].set_index('title')['movieId']
map_series

title
Confessions of a Dangerous Mind (2002)      6003
Eros (2004)                               147002
Saturn 3 (1980)                           168358
Name: movieId, dtype: int64

In [810]:
mapping_dict = {}
for title, movie_ids in idx_list.items():
    dict_value = map_series[title]
    for id in movie_ids:
        mapping_dict[id] = dict_value

In [811]:
mapping_dict

{6003: 6003,
 144606: 6003,
 32600: 147002,
 147002: 147002,
 2851: 168358,
 168358: 168358}

In [812]:
# 중복 제거 확인 (1)
ratings_df[ratings_df['movieId'].isin(drop_movieId)]

,userId,movieId,rating,timestamp
282,3,2851,5.0,1306463925
17819,111,144606,4.0,1517441257
31391,217,2851,3.0,955942393
42665,288,2851,2.0,1020369199
72810,469,2851,3.0,965335989
98357,606,32600,3.5,1171410642


In [813]:
ratings_refine = ratings_df['movieId'].map(mapping_dict) # 중복 영화에 대한 movieId 전처리 반영

In [814]:
ratings_refine = ratings_refine.fillna(ratings_df['movieId'])

In [815]:
# 정제 데이터를 기존 데이터로 변경
ratings_df['movieId'] = ratings_refine

In [816]:
# 중복 제거 확인 (2) - (1) 과 비교
ratings_df[ratings_df['movieId'].isin(drop_movieId)]

,userId,movieId,rating,timestamp


In [817]:
# 중복 제거 확인 (1) → 현재 데이터에서는 문제될 데이터 존재하지 않음
tags_df[tags_df['movieId'].isin(drop_movieId)]

,userId,movieId,tag,timestamp


In [818]:
tags_refine = tags_df['movieId'].map(mapping_dict)
tags_refine.fillna(tags_df['movieId'])
tags_df['movieId'] = tags_refine
tags_df[tags_df['movieId'].isin(drop_movieId)] # 중복 제거 확인 (2) - (1)과 비교

,userId,movieId,tag,timestamp


In [819]:
# 중복 제거 확인 (1)
links_df[links_df['movieId'].isin(drop_movieId)]

,movieId,imdbId,tmdbId
2141,2851,81454,NaN
5854,32600,377059,NaN
9106,144606,270288,4912.0


In [820]:
links_refine = links_df['movieId'].map(mapping_dict)
links_refine = links_refine.fillna(links_df['movieId'])
links_refine

0            1.0
1            2.0
2            3.0
3            4.0
4            5.0
          ...   
9737    193581.0
9738    193583.0
9739    193585.0
9740    193587.0
9741    193609.0
Name: movieId, Length: 9742, dtype: float64

In [821]:
links_df['movieId'] = links_refine
links_df[links_df['movieId'].isin(drop_movieId)] # 중복 제거 확인 (2) - (1) 과 비교

,movieId,imdbId,tmdbId


In [822]:
print('movie_df 의 중복치 제거 확인 (0 이면 제거 완료) :\t', movies_df['movieId'].isin(drop_movieId).sum())
print('rating_df 의 중복치 제거 확인 :\t',ratings_df['movieId'].isin(drop_movieId).sum())
print('tags_df 의 중복치 제거 확인 :\t', tags_df['movieId'].isin(drop_movieId).sum())
print('links_df 의 중복치 제거 확인 :\t', links_df['movieId'].isin(drop_movieId).sum())

movie_df 의 중복치 제거 확인 (0 이면 제거 완료) :	 0
rating_df 의 중복치 제거 확인 :	 0
tags_df 의 중복치 제거 확인 :	 0
links_df 의 중복치 제거 확인 :	 0


# e) year data check
- 재개봉 혹은 리메이크 영화에 대한 처리
- 제작연도가 명시되어 있지 않은 영화에 대한 처리

In [823]:
print(len(movies_df[~movies_df['title'].str.contains(r'\(')]), '개의 연도 결측치 존재')
temp_for_year = movies_df[~movies_df['title'].str.contains(r'\(')]
temp_for_year

12 개의 연도 결측치 존재


,movieId,title,genres
6059,40697,Babylon 5,[Sci-Fi]
9031,140956,Ready Player One,"[Action, Sci-Fi, Thriller]"
9091,143410,Hyena Road,"[War, Drama, Thriller, Action]"
9138,147250,The Adventures of Sherlock Holmes and Doctor Watson,"[Mystery, Crime, Adventure]"
9179,149334,Nocturnal Animals,"[Drama, Thriller]"
9259,156605,Paterson,"[Drama, Comedy, Romance]"
9367,162414,Moonlight,[Drama]
9448,167570,The OA,"[Mystery, Drama]"
9514,171495,Cosmos,[Documentary]
9515,171631,Maria Bamford: Old Baby,"[Comedy, Documentary]"


In [824]:
temp_for_year['movieId']

6059     40697
9031    140956
9091    143410
9138    147250
9179    149334
9259    156605
9367    162414
9448    167570
9514    171495
9515    171631
9525    171891
9611    176601
Name: movieId, dtype: int64

In [825]:
links_df['movieId'] = links_df['movieId'].astype('int64')

In [826]:
links_df[links_df['movieId'].isin(temp_for_year['movieId'])]

,movieId,imdbId,tmdbId
6059,40697,105946,NaN
9031,140956,1677720,333339.0
9091,143410,4034452,316042.0
9138,147250,229922,127605.0
9179,149334,4550098,340666.0
9259,156605,5247022,370755.0
9367,162414,4975722,376867.0
9448,167570,4635282,432192.0
9514,171495,81846,409926.0
9515,171631,6264596,455601.0


# imdbId 토대로 연도 조사

Babylon 5 (1993)
Ready Player One (2018)
Hyena Road (2015)
The Adventures of Sherlock Holmes and Doctor Watson (1980)
Nocturnal Animals (2016)
Paterson (2016)
Moonlight (2016)
The OA (2016)
Cosmos (1980)
Maria Bamford: Old Baby	(2017)
Generation Iron 2 (2017)
Black Mirror (2011)

In [827]:
year_list = [1993, 2018, 2015, 1980, 2016, 2016, 2016, 2016, 1980, 2017, 2017, 2011]
i=0
for title in temp_for_year['title'].values: # 연도 결측치 영화에 대해 title 을 순회
    movies_df.loc[movies_df['title']==title, 'title'] = f"{title} ({year_list[i]})"
    i+=1
    
movies_df.loc[temp_for_year.index] # temp_for_year 를 이용해 변경 체크 → reset_index 이전이기 때문에 인덱스로 접근

,movieId,title,genres
6059,40697,Babylon 5 (1993),[Sci-Fi]
9031,140956,Ready Player One (2018),"[Action, Sci-Fi, Thriller]"
9091,143410,Hyena Road (2015),"[War, Drama, Thriller, Action]"
9138,147250,The Adventures of Sherlock Holmes and Doctor Watson (1980),"[Mystery, Crime, Adventure]"
9179,149334,Nocturnal Animals (2016),"[Drama, Thriller]"
9259,156605,Paterson (2016),"[Drama, Comedy, Romance]"
9367,162414,Moonlight (2016),[Drama]
9448,167570,The OA (2016),"[Mystery, Drama]"
9514,171495,Cosmos (1980),[Documentary]
9515,171631,Maria Bamford: Old Baby (2017),"[Comedy, Documentary]"


In [828]:
print(len(movies_df[~movies_df['title'].str.contains(r'\(')]), '개의 연도 결측치 존재')
temp_for_year = movies_df[~movies_df['title'].str.contains(r'\(')]
temp_for_year

0 개의 연도 결측치 존재


,movieId,title,genres


# reset index

In [829]:
movies_df

,movieId,title,genres
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]"
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]"
2,3,Grumpier Old Men (1995),"[Comedy, Romance]"
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]"
4,5,Father of the Bride Part II (1995),[Comedy]
...,...,...,...
9737,193581,Black Butler: Book of the Atlantic (2017),"[Action, Animation, Comedy, Fantasy]"
9738,193583,No Game No Life: Zero (2017),"[Animation, Comedy, Fantasy]"
9739,193585,Flint (2017),[Drama]
9740,193587,Bungo Stray Dogs: Dead Apple (2018),"[Action, Animation]"


In [830]:
movies_df = movies_df.reset_index(drop=True)
movies_df

,movieId,title,genres
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]"
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]"
2,3,Grumpier Old Men (1995),"[Comedy, Romance]"
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]"
4,5,Father of the Bride Part II (1995),[Comedy]
...,...,...,...
9734,193581,Black Butler: Book of the Atlantic (2017),"[Action, Animation, Comedy, Fantasy]"
9735,193583,No Game No Life: Zero (2017),"[Animation, Comedy, Fantasy]"
9736,193585,Flint (2017),[Drama]
9737,193587,Bungo Stray Dogs: Dead Apple (2018),"[Action, Animation]"


In [831]:
ratings_df

,userId,movieId,rating,timestamp
0,1,1.0,4.0,964982703
1,1,3.0,4.0,964981247
2,1,6.0,4.0,964982224
3,1,47.0,5.0,964983815
4,1,50.0,5.0,964982931
...,...,...,...,...
100831,610,166534.0,4.0,1493848402
100832,610,168248.0,5.0,1493850091
100833,610,168250.0,5.0,1494273047
100834,610,168252.0,5.0,1493846352


# Year split

In [832]:
movies_df['title'] = movies_df['title'].str.strip()
movies_df

,movieId,title,genres
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]"
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]"
2,3,Grumpier Old Men (1995),"[Comedy, Romance]"
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]"
4,5,Father of the Bride Part II (1995),[Comedy]
...,...,...,...
9734,193581,Black Butler: Book of the Atlantic (2017),"[Action, Animation, Comedy, Fantasy]"
9735,193583,No Game No Life: Zero (2017),"[Animation, Comedy, Fantasy]"
9736,193585,Flint (2017),[Drama]
9737,193587,Bungo Stray Dogs: Dead Apple (2018),"[Action, Animation]"


In [833]:
movies_df[['title_only', 'year']] = movies_df['title'].str.rsplit(r' ', n=1, expand=True) # expand means DataFrame or Series
movies_df

,movieId,title,genres,title_only,year
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",Toy Story,(1995)
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",Jumanji,(1995)
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",Grumpier Old Men,(1995)
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",Waiting to Exhale,(1995)
4,5,Father of the Bride Part II (1995),[Comedy],Father of the Bride Part II,(1995)
...,...,...,...,...,...
9734,193581,Black Butler: Book of the Atlantic (2017),"[Action, Animation, Comedy, Fantasy]",Black Butler: Book of the Atlantic,(2017)
9735,193583,No Game No Life: Zero (2017),"[Animation, Comedy, Fantasy]",No Game No Life: Zero,(2017)
9736,193585,Flint (2017),[Drama],Flint,(2017)
9737,193587,Bungo Stray Dogs: Dead Apple (2018),"[Action, Animation]",Bungo Stray Dogs: Dead Apple,(2018)


In [834]:
title_duplicated_df = movies_df[movies_df['title_only'].duplicated()] # only title
title_duplicated_df

,movieId,title,genres,title_only,year
697,915,Sabrina (1954),"[Comedy, Romance]",Sabrina,(1954)
1032,1344,Cape Fear (1962),"[Crime, Drama, Thriller]",Cape Fear,(1962)
1369,1873,"Misérables, Les (1998)","[Crime, Drama, Romance, War]","Misérables, Les",(1998)
1419,1941,Hamlet (1948),[Drama],Hamlet,(1948)
1527,2059,"Parent Trap, The (1998)","[Children, Comedy, Romance]","Parent Trap, The",(1998)
...,...,...,...,...,...
9603,176413,Bliss (2012),[Drama],Bliss,(2012)
9617,177763,Murder on the Orient Express (2017),"[Crime, Drama, Mystery]",Murder on the Orient Express,(2017)
9688,184349,Elsa & Fred (2005),"[Comedy, Drama, Romance]",Elsa & Fred,(2005)
9693,184931,Death Wish (2018),"[Action, Crime, Drama, Thriller]",Death Wish,(2018)


# check year duplicated

In [835]:
title_year_list = title_duplicated_df.groupby(by='title_only')['year'].apply(list)
title_year_list[title_year_list.apply(len) > 2]

title_only
Christmas Carol, A               [(2009), (1999), (1977)]
Hamlet                   [(1948), (1964), (2000), (1990)]
Jane Eyre                        [(1944), (1970), (2011)]
Misérables, Les                  [(1998), (2012), (2000)]
Three Musketeers, The            [(1948), (1973), (2011)]
Name: year, dtype: object